# 13 — GPT-5-mini sull'intero dataset (Azure OpenAI Batch API)

Genera i riassunti di **tutte le 56.101 righe** di `complete.tab` con GPT-5-mini tramite la
**Batch API** di Azure OpenAI: sconto del 50% sui token (~40 $ per l'intera corsa) e nessun
rate limit da gestire, in cambio di un flusso asincrono (invio → attesa fino a 24 h → raccolta).
La Batch API è disponibile **solo** per i modelli Azure OpenAI: per questo Claude e DeepSeek
(notebook 11–12) si fermano alla split test.

Il notebook è **a stadi**, ciascuno rieseguibile:

1. **Costruzione dei chunk JSONL** — una richiesta per riga mancante (`custom_id` = `row_id`),
   spezzata in file da ~180 MB per rispettare i limiti Azure (~200 MB / 100.000 richieste per
   file). I chunk e lo stato dei job vivono in `results/batch/` (gitignorato).
2. **Invio** — carica ogni chunk (`purpose='batch'`) e crea il job batch sul deployment
   **Global-Batch**; gli ID dei job finiscono nel file di stato, quindi il notebook può essere
   chiuso e riaperto.
3. **Raccolta** — per i job completati scarica l'output, importa i riassunti nel TSV standard
   `results/summaries/gpt5mini_full.tsv` (righe fallite/vuote restano assenti e ritentabili
   ricominciando dallo stadio 1).
4. **Valutazione** — metriche standard in streaming su `complete.tab`.

Prompt e parametri identici al notebook 10 (modello con reasoning: niente `temperature`,
`max_completion_tokens=1500` con `reasoning_effort='minimal'` — vedi le deviazioni documentate
lì). Prerequisiti: deployment **Global-Batch** di `gpt-5-mini` e le stesse variabili d'ambiente
del notebook 10. ⚠️ La quota di *enqueued tokens*
del deployment può limitare quanti chunk possono essere in coda insieme: la cella di invio si
ferma al primo rifiuto e si rilancia quando i job precedenti sono finiti.

Per una **prova end-to-end economica** impostare `LIMIT_RICHIESTE = 10` nella configurazione:
costruisce un unico chunk da 10 righe da far passare per tutti gli stadi.

## Ripresa e rischio di mescolare corse

⚠️ Lo stadio 1 salta i `row_id` già presenti in `gpt5mini_full.tsv`, quindi ricostruire e
rinviare i chunk completa una corsa interrotta. Come per gli altri notebook, rigenerare sopra un
TSV prodotto con un deployment o una configurazione diversi mescolerebbe due corse: in quel caso
eliminare prima il TSV (e svuotare `results/batch/`).

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [ ]:
# --- Configurazione ---------------------------------------------------------
import json
import os
import summ_utils as su

METODO = 'gpt5mini'
SCOPE  = 'full'
LIMIT_RICHIESTE = None   # es. 10 per una prova end-to-end economica; None = tutte le righe

MODELLO          = 'gpt-5-mini'
DEPLOYMENT_BATCH = 'gpt-5-mini-batch'    # nome del deployment Global-Batch nel portale Azure
AZURE_ENDPOINT   = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_API_KEY    = os.environ['AZURE_OPENAI_API_KEY']
# API "v1" di Azure OpenAI: nessuna api-version datata (le vecchie preview sono state ritirate)
BASE_URL = AZURE_ENDPOINT.rstrip('/') + '/openai/v1/'

MAX_FILE_MB = 180        # limite Azure ~200 MB per file di input: teniamo margine
# Modello con reasoning: niente temperature, budget condiviso con i token di reasoning
MAX_COMPLETION_TOKENS = 1500
REASONING_EFFORT      = 'minimal'
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = 'Summarize the following document into a comprehensive summary: {documento}'

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
BATCH_DIR  = BASE / 'results' / 'batch'
STATE_PATH = BATCH_DIR / f'{METODO}_{SCOPE}_state.json'
OUT_PATH   = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

config = {'modello': MODELLO, 'deployment': DEPLOYMENT_BATCH,
          'backend': 'Azure OpenAI Batch API, API v1 (sconto 50%)',
          'max_completion_tokens': MAX_COMPLETION_TOKENS,
          'reasoning_effort': REASONING_EFFORT,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': ('prompt zero-shot identico ai notebook 07-12; modello con reasoning '
                   '(deviazioni come nel notebook 10); intera complete.tab via batch')}

print(f'Deployment batch : {DEPLOYMENT_BATCH} via {BASE_URL}')
print(f'Output           : {OUT_PATH}')
print(f'Stato job        : {STATE_PATH}')

## Stadio 1 — Costruzione dei chunk JSONL

Scorre `complete.tab` in streaming, salta i `row_id` già presenti nel TSV e scrive le richieste
in chunk da al massimo `MAX_FILE_MB`. Rifiuta di ricostruire i chunk se ci sono job ancora in
coda o in esecuzione (prima raccogliere con lo stadio 3).

In [ ]:
fatti = su.carica_riassunti(OUT_PATH)
BATCH_DIR.mkdir(parents=True, exist_ok=True)

stato = {'chunks': []}
if STATE_PATH.exists():
    stato = json.loads(STATE_PATH.read_text(encoding='utf-8'))

TERMINALI = ('importato', 'completed', 'failed', 'expired', 'cancelled')
in_corso = [c for c in stato['chunks']
            if c.get('batch_id') and c.get('stato') not in TERMINALI]
assert not in_corso, (f'{len(in_corso)} job batch ancora in corso: eseguire prima lo stadio 3 '
                      '(raccolta) e attendere che finiscano')

MAX_BYTES = MAX_FILE_MB * 1024 * 1024
chunks, corrente, scritte = [], None, 0

def chiudi_corrente():
    if corrente is not None:
        corrente['file'].close()
        chunks.append(corrente)

for es in su.itera_complete_tab(P['complete_tab']):
    if es['row_id'] in fatti:
        continue
    if LIMIT_RICHIESTE is not None and scritte >= LIMIT_RICHIESTE:
        break
    richiesta = {
        'custom_id': str(es['row_id']),
        'method': 'POST',
        'url': '/chat/completions',
        'body': {'model': DEPLOYMENT_BATCH,
                 'messages': [{'role': 'system', 'content': PROMPT_SYSTEM},
                              {'role': 'user',
                               'content': PROMPT_USER.format(
                                   documento=su.prepara_documento(es['document']))}],
                 'max_completion_tokens': MAX_COMPLETION_TOKENS,
                 'reasoning_effort': REASONING_EFFORT}}
    riga = json.dumps(richiesta, ensure_ascii=False) + '\n'
    nbytes = len(riga.encode('utf-8'))
    if corrente is None or corrente['bytes'] + nbytes > MAX_BYTES:
        chiudi_corrente()
        path = BATCH_DIR / f'chunk_{len(chunks):03d}.jsonl'
        corrente = {'path': path, 'file': open(path, 'w', encoding='utf-8'),
                    'bytes': 0, 'righe': 0}
    corrente['file'].write(riga)
    corrente['bytes'] += nbytes
    corrente['righe'] += 1
    scritte += 1
chiudi_corrente()

stato['chunks'] = [{'path': c['path'].name, 'righe': c['righe'], 'bytes': c['bytes'],
                    'batch_id': None, 'stato': 'da_inviare'} for c in chunks]
STATE_PATH.write_text(json.dumps(stato, indent=2), encoding='utf-8')
print(f'{len(chunks)} chunk creati, {scritte} richieste totali '
      f'({len(fatti)} row_id gia\' presenti nel TSV, saltati)')

## Stadio 2 — Invio dei job batch

Carica ogni chunk non ancora inviato e crea il job. Al primo rifiuto (tipicamente quota di
*enqueued tokens* esaurita) si ferma: rilanciare la cella quando i job precedenti sono stati
raccolti.

In [ ]:
from openai import OpenAI

# Rotta v1 di Azure OpenAI: copre anche files e batches, senza api-version datata
client = OpenAI(base_url=BASE_URL, api_key=AZURE_API_KEY)

stato = json.loads(STATE_PATH.read_text(encoding='utf-8'))
for chunk in stato['chunks']:
    if chunk.get('batch_id'):
        continue
    try:
        with open(BATCH_DIR / chunk['path'], 'rb') as f:
            caricato = client.files.create(file=f, purpose='batch')
        job = client.batches.create(input_file_id=caricato.id,
                                    endpoint='/chat/completions',
                                    completion_window='24h')
    except Exception as exc:
        print(f"{chunk['path']}: invio rifiutato ({exc}); mi fermo qui — "
              'rilanciare la cella quando i job in coda saranno finiti')
        break
    chunk['input_file_id'] = caricato.id
    chunk['batch_id'] = job.id
    chunk['stato'] = job.status
    STATE_PATH.write_text(json.dumps(stato, indent=2), encoding='utf-8')
    print(f"{chunk['path']}: batch {job.id} ({job.status}, {chunk['righe']} richieste)")

## Stadio 3 — Raccolta dei risultati

Interroga lo stato dei job e, per quelli completati, scarica l'output e importa i riassunti nel
TSV standard tramite `ScrittoreRiassunti` (i risultati arrivano in ordine sparso: si usa
`custom_id`). Righe vuote o fallite restano assenti dal TSV: si ritentano ricominciando dallo
stadio 1. Rieseguire la cella finché tutti i job risultano `importato`.

In [ ]:
stato = json.loads(STATE_PATH.read_text(encoding='utf-8'))
scrittore = su.ScrittoreRiassunti(OUT_PATH)

for chunk in stato['chunks']:
    if not chunk.get('batch_id') or chunk['stato'] == 'importato':
        continue
    job = client.batches.retrieve(chunk['batch_id'])
    chunk['stato'] = job.status
    print(f"{chunk['path']}: {job.status} ({job.request_counts})")
    if job.status == 'completed' and job.output_file_id:
        contenuto = client.files.content(job.output_file_id).text
        nuovi, saltati = 0, 0
        for riga in contenuto.splitlines():
            ris = json.loads(riga)
            row_id = int(ris['custom_id'])
            if scrittore.gia_fatto(row_id):
                continue
            corpo = (ris.get('response') or {}).get('body') or {}
            scelte = corpo.get('choices') or []
            testo = ((scelte[0].get('message') or {}).get('content') or '').strip() \
                if scelte else ''
            if not testo:
                saltati += 1   # errore o risposta vuota: resta assente, ritentabile
                continue
            scrittore.scrivi(row_id, testo)
            nuovi += 1
        chunk['stato'] = 'importato'
        print(f'  importati {nuovi} riassunti ({saltati} vuoti/falliti, ritentabili)')
    STATE_PATH.write_text(json.dumps(stato, indent=2), encoding='utf-8')

scrittore.chiudi()
print(f'Totale riassunti in {OUT_PATH.name}: {len(su.carica_riassunti(OUT_PATH))}')

## Stadio 4 — Valutazione (indipendente dalla generazione)

Metriche standard lette in streaming da `complete.tab`; valuta solo i `row_id` presenti nel TSV,
quindi è eseguibile anche a corsa parziale. Output: `gpt5mini_full_per_example.csv` e
`gpt5mini_full_aggregate.json`.

In [ ]:
riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.itera_complete_tab(P['complete_tab'])

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")